In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
import os
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from skimage import morphology, measure
from skimage.transform import resize
from sklearn.cluster import KMeans
import matplotlib.patches as patches
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split



In [2]:
train_csv = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")



In [3]:
base_week = train_csv.groupby("Patient")["Weeks"].min()
train_csv["base_week"] = train_csv["Patient"].map(base_week)

train_csv["count_from_base_week"] = train_csv["Weeks"] - train_csv["Patient"].map(
    base_week
)

train_csv["confidence"] = 0.0

base_fvc_dict = {
    pid: train_csv[(train_csv["Patient"] == pid) & (train_csv["Weeks"] == bw)][
        "FVC"
    ].values[0]
    for pid, bw in base_week.items()
}
train_csv["base_fvc"] = train_csv["Patient"].map(base_fvc_dict)

base_fev1_dict = {}
for pid in train_csv["Patient"].unique():
    A = train_csv[train_csv["Patient"] == pid]["base_fvc"].iloc[0]
    B = train_csv[train_csv["Patient"] == pid]["Age"].iloc[0]
    if train_csv[train_csv["Patient"] == pid]["Sex"].iloc[0] == "Male":
        base_fev1_dict[pid] = 0.77 * A + 0.32 + 0.0069 * B
    else:
        base_fev1_dict[pid] = 0.77 * A + 0.28 + 0.0052 * B
train_csv["base_fev1"] = train_csv["Patient"].map(base_fev1_dict)

base_week_percent = {
    pid: train_csv[(train_csv["Patient"] == pid) & (train_csv["Weeks"] == bw)][
        "Percent"
    ].values[0]
    for pid, bw in base_week.items()
}
train_csv["base_week_percent"] = train_csv["Patient"].map(base_week_percent)

train_csv["base fev1/base fvc"] = train_csv["base_fev1"] / train_csv["base_fvc"]
train_csv["base_height"] = (train_csv["base_fvc"] + 9030) / 77.0

base_weight_dict = {}
for pid in train_csv["Patient"].unique():
    FVC = train_csv[train_csv["Patient"] == pid]["base_fvc"].iloc[0]
    A = train_csv[train_csv["Patient"] == pid]["Age"].iloc[0]
    H = train_csv[train_csv["Patient"] == pid]["base_height"].iloc[0]
    if train_csv[train_csv["Patient"] == pid]["Sex"].iloc[0] == "Male":
        base_weight_dict[pid] = (FVC + 5458 - 49 * H + 8 * A) / 12.0
    else:
        base_weight_dict[pid] = (FVC + 3863 - 37 * H + 6 * A) / 14.0
train_csv["base_weight"] = train_csv["Patient"].map(base_weight_dict)
train_csv["base_bmi"] = train_csv["base_weight"] / (
    (train_csv["base_height"] / 100) ** 2
)



In [4]:
lb_sex = LabelEncoder()
lb_smoke = LabelEncoder()
train_csv["Sex"] = lb_sex.fit_transform(train_csv["Sex"])
train_csv["SmokingStatus"] = lb_smoke.fit_transform(train_csv["SmokingStatus"])



In [5]:
features_to_scale = [
    "Weeks",
    "Age",
    "base_week",
    "count_from_base_week",
    "base_fvc",
    "base_fev1",
    "base_week_percent",
    "base fev1/base fvc",
    "base_height",
    "base_weight",
    "base_bmi",
    "Sex",
    "SmokingStatus",
]
sc = StandardScaler()
train_scaled_vals = sc.fit_transform(train_csv[features_to_scale])
train_scaled = pd.DataFrame(train_scaled_vals, columns=features_to_scale)



In [6]:
sub = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/sample_submission.csv")
test_csv = pd.read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")



In [7]:
test_week = sub["Patient_Week"].apply(lambda x: int(x.split("_")[-1]))
patient_id = sub["Patient_Week"].apply(lambda x: x.rsplit("_", 1)[0])
sub["Patient"] = patient_id
sub["Weeks"] = test_week
sub = sub.drop(columns=["FVC", "Confidence"])

base_fvc_test = test_csv.groupby("Patient")["FVC"].min()
sub["base_fvc"] = sub["Patient"].map(base_fvc_test)

base_fev1_test = {}
for pid in sub["Patient"].unique():
    A = sub[sub["Patient"] == pid]["base_fvc"].iloc[0]
    B = test_csv[test_csv["Patient"] == pid]["Age"].iloc[0]
    sex_val = test_csv[test_csv["Patient"] == pid]["Sex"].iloc[0]
    if sex_val == "Male":
        base_fev1_test[pid] = 0.77 * A + 0.32 + 0.0069 * B
    else:
        base_fev1_test[pid] = 0.77 * A + 0.28 + 0.0052 * B
sub["base_fev1"] = sub["Patient"].map(base_fev1_test)

sub["base_height"] = (sub["base_fvc"] + 9030) / 77.0

base_weight_test = {}
for pid in sub["Patient"].unique():
    FVC = sub[sub["Patient"] == pid]["base_fvc"].iloc[0]
    A = test_csv[test_csv["Patient"] == pid]["Age"].iloc[0]
    H = sub[sub["Patient"] == pid]["base_height"].iloc[0]
    sex_val = test_csv[test_csv["Patient"] == pid]["Sex"].iloc[0]
    if sex_val == "Male":
        base_weight_test[pid] = (FVC + 5458 - 49 * H + 8 * A) / 12.0
    else:
        base_weight_test[pid] = (FVC + 3863 - 37 * H + 6 * A) / 14.0
sub["base_weight"] = sub["Patient"].map(base_weight_test)

sub["base_bmi"] = sub["base_weight"] / ((sub["base_height"] / 100) ** 2)

age_dict = test_csv.set_index("Patient")["Age"].to_dict()
sub["Age"] = sub["Patient"].map(age_dict)

base_week_test = test_csv.groupby("Patient")["Weeks"].min()
sub["base_week"] = sub["Patient"].map(base_week_test)
sub["count_from_base_week"] = sub["Weeks"] - sub["base_week"]

base_week_percent_test = {}
for pid, bw in base_week_test.items():
    perc = test_csv[(test_csv["Patient"] == pid) & (test_csv["Weeks"] == bw)][
        "Percent"
    ].values[0]
    base_week_percent_test[pid] = perc
sub["base_week_percent"] = sub["Patient"].map(base_week_percent_test)

sub["base fev1/base fvc"] = sub["base_fev1"] / sub["base_fvc"]

sub["Sex"] = lb_sex.transform(test_csv.set_index("Patient").loc[sub["Patient"], "Sex"])
sub["SmokingStatus"] = lb_smoke.transform(
    test_csv.set_index("Patient").loc[sub["Patient"], "SmokingStatus"]
)



In [8]:
test_features = [
    "Weeks",
    "Age",
    "base_week",
    "count_from_base_week",
    "base_fvc",
    "base_fev1",
    "base_week_percent",
    "base fev1/base fvc",
    "base_height",
    "base_weight",
    "base_bmi",
    "Sex",
    "SmokingStatus",
]
sub_scaled_vals = sc.transform(sub[test_features])
sub_scaled = pd.DataFrame(sub_scaled_vals, columns=test_features)



In [9]:
X = train_scaled.values
y_fvc = train_csv["FVC"].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y_fvc, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)


def metric(actual_fvc, predicted_fvc, confidence):
    sd_clipped = np.maximum(confidence, 70)
    delta = np.minimum(np.abs(actual_fvc - predicted_fvc), 1000)
    return np.mean(-np.sqrt(2) * delta / sd_clipped - np.log(np.sqrt(2) * sd_clipped))


# Validation predictions
val_pred = rf.predict(X_val)

# ---- New confidence logic: base it on validation residuals ----
val_residuals = np.abs(y_val - val_pred)
# Scale residuals slightly upward to stay safely above the 70 floor
conf_val = np.clip(val_residuals * 1.2, 70, None)

val_score = metric(y_val, val_pred, conf_val)
print(f"Validation metric (higher is better): {val_score:.4f}")

# Compute a single constant confidence to use for the test set
# (mean of the adapted validation confidences)
confidence_constant = max(70, conf_val.mean())
print(f"Derived confidence constant for test set: {confidence_constant:.2f}")



Validation metric (higher is better): -6.0875
Derived confidence constant for test set: 156.64


In [10]:
test_pred_fvc = rf.predict(sub_scaled.values)

sub["FVC"] = test_pred_fvc
sub["Confidence"] = confidence_constant

submission = sub[["Patient_Week", "FVC", "Confidence"]]



In [11]:
submission.to_csv("submission.csv", index=False)
print("submission.csv written, shape:", submission.shape)

submission.csv written, shape: (1908, 3)
